# Código complementar


**Arquivos esperados em `/content`:** `adult.data`, `adult.test`, `DecisionTree (1).csv`, `GaussianNB (1).csv`, `KNN (1).csv`, `LinearSVM (1).csv`, `LogisticRegression (1).csv` e `MLP.csv`.


In [ ]:
# 1. Configuração e imports
import os
import zipfile
import warnings
import joblib
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from sklearn.base import clone
from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer
from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    balanced_accuracy_score,
    confusion_matrix,
    classification_report,
    ConfusionMatrixDisplay
)
from sklearn.model_selection import GridSearchCV, StratifiedKFold
from sklearn.neighbors import KNeighborsClassifier
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.tree import DecisionTreeClassifier

warnings.filterwarnings("ignore")

BASE_DIR = "/content"
OUT_DIR = "/content/resultados"

os.makedirs(OUT_DIR, exist_ok=True)

RANDOM_STATE = 42
CV_FOLDS = 10
N_JOBS = -1


USAR_AMOSTRA_KNN = False
TAMANHO_AMOSTRA_KNN = 5000

def caminho(nome_arquivo):
    return os.path.join(BASE_DIR, nome_arquivo)

def salvar_csv(df, nome_arquivo, index=False):
    df.to_csv(os.path.join(OUT_DIR, nome_arquivo), index=index, encoding="utf-8-sig")

print("Pasta de entrada:", BASE_DIR)
print("Pasta de saída:", OUT_DIR)


## 2. Carregamento do Adult Dataset


In [ ]:
# 2. Carregar o dataset Adult
COLUNAS = [
    "age", "workclass", "fnlwgt", "education", "education_num",
    "marital_status", "occupation", "relationship", "race", "sex",
    "capital_gain", "capital_loss", "hours_per_week", "native_country", "income"
]

def limpar_textos(df):
    df = df.copy()

    for col in df.select_dtypes(include="object").columns:
        df[col] = df[col].astype(str).str.strip()
        df[col] = df[col].replace(["?", "nan", "None", ""], np.nan)

    df["income"] = (
        df["income"]
        .astype(str)
        .str.strip()
        .str.replace(".", "", regex=False)
    )

    return df

adult_data = caminho("adult.data")
adult_test = caminho("adult.test")

if not os.path.exists(adult_data):
    raise FileNotFoundError("O arquivo adult.data não foi encontrado em /content.")
if not os.path.exists(adult_test):
    raise FileNotFoundError("O arquivo adult.test não foi encontrado em /content.")

df_train = pd.read_csv(
    adult_data,
    header=None,
    names=COLUNAS,
    na_values="?",
    skipinitialspace=True
)

df_test = pd.read_csv(
    adult_test,
    header=None,
    names=COLUNAS,
    na_values="?",
    skipinitialspace=True,
    skiprows=1
)

df_train = limpar_textos(df_train)
df_test = limpar_textos(df_test)

X_train = df_train.drop(columns=["income"])
y_train = (df_train["income"] == ">50K").astype(int)

X_test = df_test.drop(columns=["income"])
y_test = (df_test["income"] == ">50K").astype(int)

df_total = pd.concat([df_train, df_test], ignore_index=True)

print("Treino:", X_train.shape)
print("Teste:", X_test.shape)
print("Total:", df_total.shape)
print("\nDistribuição no treino:")
print(y_train.value_counts())
print("\nDistribuição no teste:")
print(y_test.value_counts())


## 3. Descrição estatística do dataset

Gerando os arquivos para caracterizar o dataset: resumo geral, estatísticas numéricas, distribuição das classes e valores ausentes.

In [ ]:
# 3. Descrição estatística do dataset
resumo_dataset = pd.DataFrame({
    "Informacao": [
        "Instancias de treino",
        "Instancias de teste",
        "Total de instancias",
        "Atributos de entrada",
        "Atributos numericos",
        "Atributos categoricos",
        "Classe negativa",
        "Classe positiva"
    ],
    "Valor": [
        len(X_train),
        len(X_test),
        len(df_total),
        X_train.shape[1],
        len(X_train.select_dtypes(include=np.number).columns),
        len(X_train.select_dtypes(exclude=np.number).columns),
        "<=50K",
        ">50K"
    ]
})

descricao_numericas = (
    df_total
    .select_dtypes(include=np.number)
    .describe()
    .T
    .reset_index()
    .rename(columns={"index": "Variavel"})
)

distribuicao_classes = df_total["income"].value_counts().reset_index()
distribuicao_classes.columns = ["Classe", "Quantidade"]
distribuicao_classes["Percentual"] = (
    distribuicao_classes["Quantidade"] / distribuicao_classes["Quantidade"].sum()
) * 100

valores_ausentes = df_total.isna().sum().reset_index()
valores_ausentes.columns = ["Coluna", "Valores_ausentes"]
valores_ausentes["Percentual"] = (
    valores_ausentes["Valores_ausentes"] / len(df_total)
) * 100

salvar_csv(resumo_dataset, "resumo_dataset.csv")
salvar_csv(descricao_numericas, "descricao_estatistica_numericas.csv")
salvar_csv(distribuicao_classes, "distribuicao_classes.csv")
salvar_csv(valores_ausentes, "valores_ausentes.csv")

plt.figure(figsize=(7, 4))
plt.bar(distribuicao_classes["Classe"], distribuicao_classes["Quantidade"])
plt.title("Distribuicao das classes no dataset Adult")
plt.xlabel("Classe")
plt.ylabel("Quantidade")
plt.tight_layout()
plt.savefig(os.path.join(OUT_DIR, "distribuicao_classes.png"), dpi=300, bbox_inches="tight")
plt.show()

display(resumo_dataset)
display(distribuicao_classes)


## 4. Funções auxiliares

Define o pré-processamento, as métricas e a geração das matrizes de confusão.

In [ ]:
# 4. Funções auxiliares
def criar_onehot_encoder():
    try:
        return OneHotEncoder(handle_unknown="ignore", sparse_output=False)
    except TypeError:
        return OneHotEncoder(handle_unknown="ignore", sparse=False)

def criar_preprocessador(X):
    colunas_numericas = X.select_dtypes(include=np.number).columns.tolist()
    colunas_categoricas = X.select_dtypes(exclude=np.number).columns.tolist()

    pipeline_numerico = Pipeline([
        ("imputer", SimpleImputer(strategy="median")),
        ("scaler", StandardScaler())
    ])

    pipeline_categorico = Pipeline([
        ("imputer", SimpleImputer(strategy="most_frequent")),
        ("onehot", criar_onehot_encoder())
    ])

    return ColumnTransformer([
        ("num", pipeline_numerico, colunas_numericas),
        ("cat", pipeline_categorico, colunas_categoricas)
    ])

def avaliar_modelo(nome_modelo, y_real, y_predito):
    matriz = confusion_matrix(y_real, y_predito, labels=[0, 1])
    tn, fp, fn, tp = matriz.ravel()

    metricas = {
        "Modelo": nome_modelo,
        "TN": tn,
        "FP": fp,
        "FN": fn,
        "TP": tp,
        "Acuracia": accuracy_score(y_real, y_predito),
        "Precisao": precision_score(y_real, y_predito, zero_division=0),
        "Recall": recall_score(y_real, y_predito, zero_division=0),
        "Especificidade": tn / (tn + fp) if (tn + fp) > 0 else 0,
        "F1_score": f1_score(y_real, y_predito, zero_division=0),
        "Balanced_Accuracy": balanced_accuracy_score(y_real, y_predito)
    }

    matriz_df = pd.DataFrame(
        matriz,
        index=["Real <=50K", "Real >50K"],
        columns=["Predito <=50K", "Predito >50K"]
    )
    salvar_csv(
        matriz_df.reset_index().rename(columns={"index": "Classe_real"}),
        f"matriz_confusao_{nome_modelo}.csv"
    )

    report_df = pd.DataFrame(
        classification_report(
            y_real,
            y_predito,
            target_names=["<=50K", ">50K"],
            output_dict=True,
            zero_division=0
        )
    ).T.reset_index().rename(columns={"index": "Classe"})
    salvar_csv(report_df, f"classification_report_{nome_modelo}.csv")

    disp = ConfusionMatrixDisplay(
        confusion_matrix=matriz,
        display_labels=["<=50K", ">50K"]
    )
    disp.plot(cmap="Blues", values_format="d")
    plt.title(f"Matriz de Confusao - {nome_modelo}")
    plt.tight_layout()
    plt.savefig(os.path.join(OUT_DIR, f"matriz_confusao_{nome_modelo}.png"), dpi=300, bbox_inches="tight")
    plt.show()

    return metricas

scoring = {
    "accuracy": "accuracy",
    "precision": "precision",
    "recall": "recall",
    "f1": "f1",
    "balanced_accuracy": "balanced_accuracy"
}

cv = StratifiedKFold(n_splits=CV_FOLDS, shuffle=True, random_state=RANDOM_STATE)


## 5. GridSearch da Árvore de Decisão

Testa os principais hiperparâmetros da Árvore de Decisão e salva o modelo final, os resultados da busca, o relatório de classificação e a matriz de confusão.

In [ ]:
# 5. GridSearch da Decision Tree
pipeline_dt = Pipeline([
    ("preprocessor", criar_preprocessador(X_train)),
    ("classifier", DecisionTreeClassifier(random_state=RANDOM_STATE))
])

parametros_dt = {
    "classifier__criterion": ["gini", "entropy"],
    "classifier__max_depth": [5, 10, 20, None],
    "classifier__min_samples_split": [2, 5, 10]
}

grid_dt = GridSearchCV(
    estimator=pipeline_dt,
    param_grid=parametros_dt,
    scoring=scoring,
    refit="f1",
    cv=cv,
    n_jobs=N_JOBS,
    return_train_score=False,
    verbose=1
)

grid_dt.fit(X_train, y_train)

modelo_final_dt = grid_dt.best_estimator_
y_pred_dt = modelo_final_dt.predict(X_test)

metricas_dt = avaliar_modelo("DecisionTree", y_test, y_pred_dt)

salvar_csv(pd.DataFrame(grid_dt.cv_results_), "cv_results_DecisionTree.csv")
joblib.dump(grid_dt, os.path.join(OUT_DIR, "GridSearch_DecisionTree.joblib"))
joblib.dump(modelo_final_dt, os.path.join(OUT_DIR, "Modelo_Final_DecisionTree.joblib"))

print("Melhores parametros - Decision Tree:")
print(grid_dt.best_params_)
print("\nMetricas no teste:")
display(pd.DataFrame([metricas_dt]))


## 6. GridSearch do KNN

Testa os principais hiperparâmetros do KNN e salva o modelo final, os resultados da busca, o relatório de classificação e a matriz de confusão.

In [ ]:
# 6. GridSearch do KNN
pipeline_knn = Pipeline([
    ("preprocessor", criar_preprocessador(X_train)),
    ("classifier", KNeighborsClassifier())
])

parametros_knn = {
    "classifier__n_neighbors": [3, 5, 7, 11],
    "classifier__weights": ["uniform", "distance"]
}

if USAR_AMOSTRA_KNN:
    from sklearn.model_selection import StratifiedShuffleSplit

    sss = StratifiedShuffleSplit(
        n_splits=1,
        train_size=min(TAMANHO_AMOSTRA_KNN, len(X_train)),
        random_state=RANDOM_STATE
    )
    idx_amostra, _ = next(sss.split(X_train, y_train))
    X_grid_knn = X_train.iloc[idx_amostra]
    y_grid_knn = y_train.iloc[idx_amostra]
    observacao_knn = f"GridSearch em amostra estratificada de {len(X_grid_knn)} registros; modelo final treinado no treino completo."
else:
    X_grid_knn = X_train
    y_grid_knn = y_train
    observacao_knn = "GridSearch com treino oficial completo."

grid_knn = GridSearchCV(
    estimator=pipeline_knn,
    param_grid=parametros_knn,
    scoring=scoring,
    refit="f1",
    cv=cv,
    n_jobs=N_JOBS,
    return_train_score=False,
    verbose=1
)

grid_knn.fit(X_grid_knn, y_grid_knn)

modelo_final_knn = clone(pipeline_knn)
modelo_final_knn.set_params(**grid_knn.best_params_)
modelo_final_knn.fit(X_train, y_train)

y_pred_knn = modelo_final_knn.predict(X_test)

metricas_knn = avaliar_modelo("KNN", y_test, y_pred_knn)

salvar_csv(pd.DataFrame(grid_knn.cv_results_), "cv_results_KNN.csv")
joblib.dump(grid_knn, os.path.join(OUT_DIR, "GridSearch_KNN.joblib"))
joblib.dump(modelo_final_knn, os.path.join(OUT_DIR, "Modelo_Final_KNN.joblib"))

print("Melhores parametros - KNN:")
print(grid_knn.best_params_)
print("\nMetricas no teste:")
display(pd.DataFrame([metricas_knn]))


## 7. Tabela final da GridSearch

Consolida Árvore de Decisão e KNN em uma tabela final com médias, desvios padrão, melhores parâmetros e métricas no teste.

In [ ]:
# 7. Tabela final da GridSearch
def resumo_gridsearch(nome_modelo, grid, metricas_teste, observacao):
    idx = grid.best_index_
    cv_results = grid.cv_results_

    return {
        "Modelo": nome_modelo,
        "Melhores_parametros": str(grid.best_params_),
        "Acuracia_media_CV": cv_results["mean_test_accuracy"][idx],
        "Acuracia_desvio_CV": cv_results["std_test_accuracy"][idx],
        "Precisao_media_CV": cv_results["mean_test_precision"][idx],
        "Precisao_desvio_CV": cv_results["std_test_precision"][idx],
        "Recall_medio_CV": cv_results["mean_test_recall"][idx],
        "Recall_desvio_CV": cv_results["std_test_recall"][idx],
        "F1_medio_CV": cv_results["mean_test_f1"][idx],
        "F1_desvio_CV": cv_results["std_test_f1"][idx],
        "Balanced_Accuracy_media_CV": cv_results["mean_test_balanced_accuracy"][idx],
        "Balanced_Accuracy_desvio_CV": cv_results["std_test_balanced_accuracy"][idx],
        "Acuracia_teste": metricas_teste["Acuracia"],
        "Precisao_teste": metricas_teste["Precisao"],
        "Recall_teste": metricas_teste["Recall"],
        "F1_teste": metricas_teste["F1_score"],
        "Balanced_Accuracy_teste": metricas_teste["Balanced_Accuracy"],
        "Observacao": observacao
    }

resumo_gridsearch_modelos = pd.DataFrame([
    resumo_gridsearch("Decision Tree", grid_dt, metricas_dt, "GridSearch com treino oficial completo."),
    resumo_gridsearch("KNN", grid_knn, metricas_knn, observacao_knn)
])

salvar_csv(resumo_gridsearch_modelos, "resumo_gridsearch_modelos.csv")

tabela_artigo_gridsearch = resumo_gridsearch_modelos.copy()
tabela_artigo_gridsearch["Acuracia_CV"] = (
    tabela_artigo_gridsearch["Acuracia_media_CV"].round(4).astype(str)
    + " +/- "
    + tabela_artigo_gridsearch["Acuracia_desvio_CV"].round(4).astype(str)
)
tabela_artigo_gridsearch["F1_CV"] = (
    tabela_artigo_gridsearch["F1_medio_CV"].round(4).astype(str)
    + " +/- "
    + tabela_artigo_gridsearch["F1_desvio_CV"].round(4).astype(str)
)

tabela_artigo_gridsearch = tabela_artigo_gridsearch[
    [
        "Modelo",
        "Melhores_parametros",
        "Acuracia_CV",
        "F1_CV",
        "Acuracia_teste",
        "Precisao_teste",
        "Recall_teste",
        "F1_teste",
        "Observacao"
    ]
]

salvar_csv(tabela_artigo_gridsearch, "tabela_artigo_gridsearch.csv")

display(tabela_artigo_gridsearch)

metricas_plot = ["Acuracia_teste", "Precisao_teste", "Recall_teste", "F1_teste", "Balanced_Accuracy_teste"]
ax = resumo_gridsearch_modelos.set_index("Modelo")[metricas_plot].plot(kind="bar", figsize=(10, 5))
plt.title("Comparacao Decision Tree x KNN apos GridSearch")
plt.ylabel("Valor da metrica")
plt.xlabel("Modelo")
plt.ylim(0, 1)
plt.xticks(rotation=0)
plt.legend(loc="upper center", bbox_to_anchor=(0.5, -0.18), ncol=5, frameon=True)
plt.tight_layout()
plt.subplots_adjust(bottom=0.28)
plt.savefig(os.path.join(OUT_DIR, "comparacao_metricas_gridsearch.png"), dpi=300, bbox_inches="tight")
plt.show()


## 8. Consolidação dos CSVs da equipe


In [ ]:
# 8. Consolidar CSVs de predição da equipe
def ler_csv_flexivel(caminho_csv):
    for enc in ["utf-8-sig", "utf-8", "latin1"]:
        try:
            return pd.read_csv(caminho_csv, encoding=enc)
        except Exception:
            pass
    return pd.read_csv(caminho_csv)

def limpar_classe(serie):
    serie = serie.astype(str).str.strip().str.replace(".", "", regex=False)
    mapa = {"<=50K": 0, ">50K": 1, "0": 0, "1": 1, "0.0": 0, "1.0": 1}
    resultado = serie.map(mapa)

    if resultado.isna().any():
        valores = sorted(serie[resultado.isna()].unique().tolist())
        raise ValueError(f"Valores de classe nao reconhecidos: {valores}")

    return resultado.astype(int)

arquivos_predicoes = {
    "Decision Tree": "DecisionTree (1).csv",
    "GaussianNB": "GaussianNB (1).csv",
    "KNN": "KNN (1).csv",
    "LinearSVM": "LinearSVM (1).csv",
    "Logistic Regression": "LogisticRegression (1).csv",
    "MLP": "MLP.csv"
}

resultados_csv = []

for modelo, arquivo in arquivos_predicoes.items():
    arq_path = caminho(arquivo)

    if not os.path.exists(arq_path):
        print(f"Ignorado, arquivo nao encontrado: {arquivo}")
        continue

    df_pred = ler_csv_flexivel(arq_path)
    df_pred.columns = [str(c).strip() for c in df_pred.columns]

    if "real" not in df_pred.columns or "predito" not in df_pred.columns:
        print(f"Ignorado, colunas real/predito ausentes: {arquivo}")
        continue

    y_real = limpar_classe(df_pred["real"])
    y_pred = limpar_classe(df_pred["predito"])

    matriz = confusion_matrix(y_real, y_pred, labels=[0, 1])
    tn, fp, fn, tp = matriz.ravel()

    resultados_csv.append({
        "Modelo": modelo,
        "Acuracia": accuracy_score(y_real, y_pred),
        "Precisao": precision_score(y_real, y_pred, zero_division=0),
        "Recall": recall_score(y_real, y_pred, zero_division=0),
        "Especificidade": tn / (tn + fp) if (tn + fp) > 0 else 0,
        "F1_score": f1_score(y_real, y_pred, zero_division=0),
        "Balanced_Accuracy": balanced_accuracy_score(y_real, y_pred)
    })

metricas_modelos_csv = pd.DataFrame(resultados_csv)

if len(metricas_modelos_csv) > 0:
    metricas_modelos_csv = metricas_modelos_csv.sort_values("F1_score", ascending=False)
    salvar_csv(metricas_modelos_csv, "metricas_modelos_csv.csv")
    display(metricas_modelos_csv)

    metricas_grafico = ["Acuracia", "Precisao", "Recall", "F1_score", "Balanced_Accuracy"]
    ax = metricas_modelos_csv.set_index("Modelo")[metricas_grafico].plot(kind="bar", figsize=(12, 5))
    plt.title("Comparacao geral dos modelos")
    plt.ylabel("Valor da metrica")
    plt.xlabel("Modelo")
    plt.ylim(0, 1)
    plt.xticks(rotation=45, ha="right")
    plt.legend(loc="upper center", bbox_to_anchor=(0.5, -0.22), ncol=5, frameon=True)
    plt.tight_layout()
    plt.subplots_adjust(bottom=0.32)
    plt.savefig(os.path.join(OUT_DIR, "comparacao_modelos_csv.png"), dpi=300, bbox_inches="tight")
    plt.show()
else:
    print("Nenhum CSV de predicao foi encontrado. A etapa de consolidacao geral foi ignorada.")


## 9. ZIP final


In [14]:
# Compactando os arquivos organizados por pastas dentro do ZIP

zip_path = "/content/resultados.zip"

grupos_arquivos = {
    "1_Descricao_estatistica_do_dataset": [
        "resumo_dataset.csv",
        "descricao_estatistica_numericas.csv",
        "distribuicao_classes.csv",
        "distribuicao_classes.png",
        "valores_ausentes.csv"
    ],

    "2_Resultados_Arvore_de_Decisao": [
        "cv_results_DecisionTree.csv",
        "classification_report_DecisionTree.csv",
        "matriz_confusao_DecisionTree.csv",
        "matriz_confusao_DecisionTree.png",
        "GridSearch_DecisionTree.joblib",
        "Modelo_Final_DecisionTree.joblib"
    ],

    "3_Resultados_KNN": [
        "cv_results_KNN.csv",
        "classification_report_KNN.csv",
        "matriz_confusao_KNN.csv",
        "matriz_confusao_KNN.png",
        "GridSearch_KNN.joblib",
        "Modelo_Final_KNN.joblib"
    ],

    "4_Comparacao_final_dos_modelos": [
        "resumo_gridsearch_modelos.csv",
        "tabela_artigo_gridsearch.csv",
        "comparacao_metricas_gridsearch.png",
        "metricas_modelos_csv.csv",
        "comparacao_modelos_csv.png"
    ]
}

with zipfile.ZipFile(zip_path, "w", zipfile.ZIP_DEFLATED) as zipf:

    # Arquivos principais na raiz do ZIP
    arquivos_raiz = [
        "README.txt",
        "checklist_arquivos_gerados.csv"
    ]

    for arquivo in arquivos_raiz:
        caminho = os.path.join(OUT_DIR, arquivo)
        if os.path.exists(caminho):
            zipf.write(caminho, arquivo)

    # Arquivos organizados em pastas
    for pasta, arquivos in grupos_arquivos.items():
        for arquivo in arquivos:
            caminho = os.path.join(OUT_DIR, arquivo)

            if os.path.exists(caminho):
                caminho_no_zip = os.path.join(pasta, arquivo)
                zipf.write(caminho, caminho_no_zip)

print("ZIP final criado em:", zip_path)

try:
    from google.colab import files
    files.download(zip_path)
except Exception:
    print("Baixe manualmente o arquivo:", zip_path)

ZIP final criado em: /content/resultados.zip


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>